In [16]:
from fastavro import parse_schema, schemaless_reader
import json
from confluent_kafka import Consumer, KafkaException
from pathlib import Path
import io


In [ ]:
SCHEMA_PATH = (
    Path(".").resolve()
    / "schemas"
    / "trade_event.avsc"
)

In [ ]:
with SCHEMA_PATH.open(encoding="utf-8") as schema_file:
        schema = parse_schema(json.load(schema_file))

In [ ]:
consumer = Consumer({
    "bootstrap.servers": "localhost:9092",
    "group.id": "equity-ops-lab-reader-1",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})

In [ ]:
consumer.subscribe(["equity-ops.trades.raw"])

In [ ]:
while True:
    msg = consumer.poll(timeout=1.0)
    if msg is None: continue

    if msg.error():
        raise KafkaException(msg.error())
    else:
        stream = io.BytesIO(msg.value())
        record = schemaless_reader(stream, schema)
        consumer.close()
        break

In [ ]:
record